# 上机实验04：回归模型与模型评估

学号：1043220122  姓名：林鑫

数据集：`沪深300_最近三年.csv`

本 notebook 参照上次作业的风格组织为“数据预处理、标准化、模型训练、预测对比、结果分析”几个部分。

In [ ]:
# P1.0 导入库与路径
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import StackingRegressor, VotingRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Microsoft YaHei")

CURRENT_DIR = Path.cwd().resolve()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "code" else CURRENT_DIR
RESULT_DIR = BASE_DIR / "result"
DATA_PATH = next((BASE_DIR / "code").glob("*.csv"))

In [ ]:
# P1.1 数据加载与基础清洗
raw_df = pd.read_csv(DATA_PATH)
raw_df.columns = ["date", "close", "open", "high", "low", "volume", "pct_change"]
df = raw_df.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df["volume"] = df["volume"].astype(str).str.replace("K", "", regex=False).str.replace(",", "", regex=False).astype(float)
df["pct_change"] = pd.to_numeric(df["pct_change"].astype(str).str.replace("%", "", regex=False), errors="coerce")

missing_before = df.isna().sum()
duplicate_count = df.duplicated().sum()
df = df.dropna().drop_duplicates().reset_index(drop=True)

print("缺失值统计：")
print(missing_before)
print("重复行数量：", duplicate_count)
df.head()

In [ ]:
# P1.2 异常值处理与特征工程
numeric_cols = ["close", "open", "high", "low", "volume", "pct_change"]
outlier_records = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_records.append({"feature": col, "outlier_count": int(outlier_count)})
    df[col] = df[col].clip(lower=lower, upper=upper)

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday + 1
df["amplitude"] = df["high"] - df["low"]
df["open_high_gap"] = df["high"] - df["open"]
df["open_low_gap"] = df["open"] - df["low"]
df["high_low_ratio"] = df["high"] / df["low"]
df["intraday_range_ratio"] = (df["high"] - df["low"]) / df["open"]
df["vol_change_ratio"] = df["volume"].pct_change()
df["ma3"] = df["close"].shift(1).rolling(3).mean()
df["ma5"] = df["close"].shift(1).rolling(5).mean()
df["ma10"] = df["close"].shift(1).rolling(10).mean()
df["vol_ma5"] = df["volume"].shift(1).rolling(5).mean()
df["ret_std5"] = df["pct_change"].shift(1).rolling(5).std()
for lag in [1, 2, 3, 5]:
    df[f"close_lag{lag}"] = df["close"].shift(lag)
    df[f"open_lag{lag}"] = df["open"].shift(lag)
    df[f"volume_lag{lag}"] = df["volume"].shift(lag)
    df[f"pct_change_lag{lag}"] = df["pct_change"].shift(lag)

model_df = df.dropna().reset_index(drop=True)
pd.DataFrame(outlier_records).sort_values("outlier_count", ascending=False)

In [ ]:
# P1.3 预处理结果可视化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
missing_before.plot(kind="bar", ax=axes[0, 0], color="#d62728")
axes[0, 0].set_title("缺失值统计")
sns.histplot(df["close"], kde=True, ax=axes[0, 1], color="#1f77b4")
axes[0, 1].set_title("收盘价分布")
sns.boxplot(data=df[["close", "open", "high", "low"]], ax=axes[1, 0])
axes[1, 0].set_title("价格特征箱线图")
axes[1, 1].plot(df["date"], df["close"], color="#2ca02c", linewidth=2)
axes[1, 1].set_title("收盘价走势")
axes[1, 1].tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(RESULT_DIR / "fig1_preprocess_overview.png", dpi=300)
plt.show()

In [ ]:
# P1.4 构造特征矩阵、划分训练集和测试集
feature_cols = [
    "open", "high", "low", "volume", "year", "month", "day", "weekday",
    "amplitude", "open_high_gap", "open_low_gap",
    "high_low_ratio", "intraday_range_ratio", "vol_change_ratio",
    "ma3", "ma5", "ma10", "vol_ma5", "ret_std5"
]
for lag in [1, 2, 3, 5]:
    feature_cols.extend([f"close_lag{lag}", f"open_lag{lag}", f"volume_lag{lag}", f"pct_change_lag{lag}"])

X = model_df[feature_cols]
y = model_df["close"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
print("训练集形状：", X_train.shape)
print("测试集形状：", X_test.shape)

In [ ]:
# P1.5 标准化处理及可视化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
scale_features = ["open", "high", "volume", "close_lag1"]
scale_compare_df = pd.concat(
    [
        X_train[scale_features].assign(stage="标准化前"),
        pd.DataFrame(X_train_scaled, columns=feature_cols)[scale_features].assign(stage="标准化后"),
    ],
    ignore_index=True,
)
scale_long_df = scale_compare_df.melt(id_vars="stage", var_name="feature", value_name="value")
plt.figure(figsize=(11, 5))
sns.boxplot(data=scale_long_df, x="feature", y="value", hue="stage")
plt.title("标准化前后特征分布对比")
plt.tight_layout()
plt.savefig(RESULT_DIR / "05_standardization_compare.png", dpi=300)
plt.show()

In [ ]:
# P1.6 构建并训练各类回归模型
def eval_model(name, model):
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    return {
        "模型": name,
        "训练集R2": round(r2_score(y_train, train_pred), 4),
        "测试集R2": round(r2_score(y_test, test_pred), 4),
        "训练集RMSE": round(math.sqrt(mean_squared_error(y_train, train_pred)), 4),
        "测试集RMSE": round(math.sqrt(mean_squared_error(y_test, test_pred)), 4),
        "训练集MAE": round(mean_absolute_error(y_train, train_pred), 4),
        "测试集MAE": round(mean_absolute_error(y_test, test_pred), 4),
        "test_pred": test_pred
    }

cv = KFold(n_splits=5, shuffle=True, random_state=42)
linear_model = LinearRegression().fit(X_train_scaled, y_train)
ridge_model = GridSearchCV(Ridge(), {"alpha": [0.01, 0.1, 1, 10, 50, 100]}, cv=cv, scoring="r2", n_jobs=-1).fit(X_train_scaled, y_train).best_estimator_
lasso_model = GridSearchCV(Lasso(max_iter=100000), {"alpha": [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]}, cv=cv, scoring="r2", n_jobs=-1).fit(X_train_scaled, y_train).best_estimator_
elastic_model = GridSearchCV(ElasticNet(max_iter=100000), {"alpha": [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5], "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]}, cv=cv, scoring="r2", n_jobs=-1).fit(X_train_scaled, y_train).best_estimator_
svr_model = GridSearchCV(SVR(kernel="linear"), {"C": [0.1, 1, 10, 50], "epsilon": [0.01, 0.05, 0.1, 0.2]}, cv=cv, scoring="r2", n_jobs=-1).fit(X_train_scaled, y_train).best_estimator_
xgb_model = XGBRegressor(objective="reg:squarederror", n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.05, reg_lambda=1.0, random_state=42)
xgb_model.fit(X_train_scaled, y_train)
voting_model = VotingRegressor([("ridge", clone(ridge_model)), ("lasso", clone(lasso_model)), ("elastic", clone(elastic_model)), ("svr", clone(svr_model))]).fit(X_train_scaled, y_train)
stacking_model = StackingRegressor([("ridge", clone(ridge_model)), ("lasso", clone(lasso_model)), ("xgb", clone(xgb_model))], final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1).fit(X_train_scaled, y_train)

results = [
    eval_model("Linear", linear_model),
    eval_model("Ridge", ridge_model),
    eval_model("Lasso", lasso_model),
    eval_model("ElasticNet", elastic_model),
    eval_model("SVR", svr_model),
    eval_model("XGBoost", xgb_model),
    eval_model("Voting", voting_model),
    eval_model("Stacking", stacking_model),
]
metrics_df = pd.DataFrame([{k: v for k, v in row.items() if k != "test_pred"} for row in results]).sort_values("测试集R2", ascending=False).reset_index(drop=True)
metrics_df

In [ ]:
# P1.7 Lasso系数与模型指标可视化
coef_df = pd.DataFrame({"feature": feature_cols, "coef": lasso_model.coef_})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).head(12).sort_values("coef")
plt.figure(figsize=(9, 6))
sns.barplot(data=coef_df, x="coef", y="feature", palette="viridis")
plt.title("Lasso回归中绝对值最大的12个系数")
plt.tight_layout()
plt.savefig(RESULT_DIR / "06_lasso_coefficients.png", dpi=300)
plt.show()

x = np.arange(len(metrics_df))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(x - width / 2, metrics_df["训练集R2"], width=width, label="训练集R2")
axes[0].bar(x + width / 2, metrics_df["测试集R2"], width=width, label="测试集R2")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_df["模型"], rotation=30)
axes[0].legend()
axes[0].set_title("模型R2对比")
axes[1].bar(x - width / 2, metrics_df["训练集RMSE"], width=width, label="训练集RMSE")
axes[1].bar(x + width / 2, metrics_df["测试集RMSE"], width=width, label="测试集RMSE")
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_df["模型"], rotation=30)
axes[1].legend()
axes[1].set_title("模型RMSE对比")
fig.tight_layout()
fig.savefig(RESULT_DIR / "07_model_comparison.png", dpi=300)
plt.show()

In [ ]:
# P1.8 各模型预测结果可视化对比
compare_pred_df = pd.DataFrame({"真实值": y_test.reset_index(drop=True)})
for row in results:
    compare_pred_df[row["模型"]] = row["test_pred"]

fig, axes = plt.subplots(4, 2, figsize=(14, 14), sharex=True)
axes = axes.flatten()
for ax, model_name in zip(axes, compare_pred_df.columns[1:]):
    ax.plot(compare_pred_df.index, compare_pred_df["真实值"], label="真实值", linewidth=1.8, color="#1D3557")
    ax.plot(compare_pred_df.index, compare_pred_df[model_name], label="预测值", linewidth=1.5, color="#E76F51", alpha=0.9)
    ax.set_title(f"{model_name}预测结果对比")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULT_DIR / "08_model_prediction_compare.png", dpi=300)
plt.show()

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
axes = axes.flatten()
for ax, model_name in zip(axes, compare_pred_df.columns[1:]):
    ax.scatter(compare_pred_df["真实值"], compare_pred_df[model_name], s=18, alpha=0.75, color="#2A9D8F")
    min_val = min(compare_pred_df["真实值"].min(), compare_pred_df[model_name].min())
    max_val = max(compare_pred_df["真实值"].max(), compare_pred_df[model_name].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="red")
    ax.set_title(f"{model_name}散点对比")
fig.tight_layout()
fig.savefig(RESULT_DIR / "09_model_prediction_scatter.png", dpi=300)
plt.show()